In [ ]:
import pymongo
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter
import pandas as pd

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

# Conexión a MongoDB
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "universia_VA"

client = pymongo.MongoClient(MONGO_URI)
db = client[DB_NAME]
interactions = db.interactions
access_hashes = db.access_hashes

# Configuración de período de análisis
DAYS_BACK = 30
end_date = datetime.now()
start_date = end_date - timedelta(days=DAYS_BACK)

print(f"✓ Conexión establecida con MongoDB")
print(f"✓ Período de análisis: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")
print(f"✓ Collections disponibles: {db.list_collection_names()}")

✓ Conexión establecida con MongoDB
✓ Período de análisis: 2026-01-11 a 2026-02-10
✓ Collections disponibles: []


In [13]:
# Vista Rápida de Datos

# Contar documentos por collection
total_interactions = interactions.count_documents({"timestamp": {"$gte": start_date, "$lte": end_date}})
total_evaluations = interactions.count_documents({
    "type": "evaluation",
    "timestamp": {"$gte": start_date, "$lte": end_date}
})
total_chats = interactions.count_documents({
    "type": "chat_interaction",
    "timestamp": {"$gte": start_date, "$lte": end_date}
})
total_hashes = access_hashes.count_documents({"created_at": {"$gte": start_date, "$lte": end_date}})

# Mostrar ejemplo de documentos
print("="*70)
print("RESUMEN DE DATOS")
print("="*70)
print(f"Total Interacciones: {total_interactions}")
print(f"  - Evaluaciones: {total_evaluations}")
print(f"  - Chats: {total_chats}")
print(f"Total Hashes de Acceso: {total_hashes}")
print()

# Muestra de una interacción de evaluación
print("Ejemplo de Evaluación:")
print("-"*70)
sample_eval = interactions.find_one({"type": "evaluation"})
if sample_eval:
    print(f"Usuario: {sample_eval.get('user_id', 'N/A')[:20]}...")
    print(f"Video: {sample_eval.get('video_url', 'N/A')}")
    print(f"Idioma: {sample_eval.get('language', 'N/A')}")
    print(f"Tiempo de respuesta: {sample_eval.get('response_time_seconds', 0):.2f}s")
    print(f"Resultado: {'APROBADO' if sample_eval.get('evaluation_result', {}).get('pass') else 'SUSPENDIDO'}")
print()

# Muestra de un chat
print("Ejemplo de Chat:")
print("-"*70)
sample_chat = interactions.find_one({"type": "chat_interaction"})
if sample_chat:
    print(f"Usuario: {sample_chat.get('user_id', 'N/A')[:20]}...")
    print(f"Pregunta: {sample_chat.get('user_input', 'N/A')[:80]}...")
    print(f"Respuesta: {sample_chat.get('model_response', 'N/A')[:80]}...")

RESUMEN DE DATOS
Total Interacciones: 0
  - Evaluaciones: 0
  - Chats: 0
Total Hashes de Acceso: 0

Ejemplo de Evaluación:
----------------------------------------------------------------------

Ejemplo de Chat:
----------------------------------------------------------------------


In [14]:
# Rendimiento en Evaluaciones

# Obtener evaluaciones del período
evaluations = list(interactions.find({
    "type": "evaluation",
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not evaluations:
    print("⚠️ No hay evaluaciones en el período seleccionado")
else:
    # Calcular métricas generales
    total_evals = len(evaluations)
    passed_evals = sum(1 for e in evaluations if e.get('evaluation_result', {}).get('pass', False))
    pass_rate = (passed_evals / total_evals) * 100
    
    # Métricas por estudiante
    student_stats = defaultdict(lambda: {'passed': 0, 'total': 0})
    for eval in evaluations:
        user_id = eval.get('user_id')
        student_stats[user_id]['total'] += 1
        if eval.get('evaluation_result', {}).get('pass', False):
            student_stats[user_id]['passed'] += 1
    
    # Evolución diaria
    daily_stats = defaultdict(lambda: {'passed': 0, 'total': 0})
    for eval in evaluations:
        date = eval['timestamp'].date()
        daily_stats[date]['total'] += 1
        if eval.get('evaluation_result', {}).get('pass', False):
            daily_stats[date]['passed'] += 1
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Rendimiento en Evaluaciones', fontsize=16, fontweight='bold')
    
    # 1. Tasa de aprobación general
    ax1 = axes[0, 0]
    ax1.bar(['Aprobados', 'Suspendidos'], 
            [passed_evals, total_evals - passed_evals],
            color=['#2ecc71', '#e74c3c'])
    ax1.set_ylabel('Número de Evaluaciones')
    ax1.set_title(f'Tasa de Aprobación General: {pass_rate:.1f}%')
    for i, v in enumerate([passed_evals, total_evals - passed_evals]):
        ax1.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    
    # 2. Distribución por estudiante
    ax2 = axes[0, 1]
    student_rates = [(stats['passed'] / stats['total']) * 100 
                     for stats in student_stats.values()]
    ax2.hist(student_rates, bins=10, color='#3498db', edgecolor='black', alpha=0.7)
    ax2.axvline(np.mean(student_rates), color='red', linestyle='--', 
                label=f'Media: {np.mean(student_rates):.1f}%')
    ax2.set_xlabel('Tasa de Aprobación (%)')
    ax2.set_ylabel('Número de Estudiantes')
    ax2.set_title('Distribución de Tasas de Aprobación')
    ax2.legend()
    
    # 3. Top estudiantes (filtrar los que tienen al menos 3 evaluaciones)
    ax3 = axes[1, 0]
    sorted_students = sorted(
        [(uid, stats) for uid, stats in student_stats.items() if stats['total'] >= 3],
        key=lambda x: x[1]['passed'] / x[1]['total'],
        reverse=True
    )[:10]
    
    if sorted_students:
        student_names = [f"Est. {i+1}" for i in range(len(sorted_students))]
        student_percentages = [(s[1]['passed'] / s[1]['total']) * 100 for s in sorted_students]
        
        ax3.barh(student_names, student_percentages, color='#9b59b6')
        ax3.set_xlabel('Tasa de Aprobación (%)')
        ax3.set_title('Top 10 Estudiantes por Rendimiento (min. 3 eval.)')
        ax3.invert_yaxis()
    
    # 4. Evolución temporal
    ax4 = axes[1, 1]
    sorted_dates = sorted(daily_stats.keys())
    daily_pass_rates = [(daily_stats[d]['passed'] / daily_stats[d]['total']) * 100 
                        for d in sorted_dates]
    
    ax4.plot(sorted_dates, daily_pass_rates, marker='o', linewidth=2, color='#e67e22')
    ax4.axhline(pass_rate, color='gray', linestyle='--', alpha=0.5, label='Media global')
    ax4.set_xlabel('Fecha')
    ax4.set_ylabel('Tasa de Aprobación (%)')
    ax4.set_title('Evolución Temporal del Rendimiento')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Rendimiento en Evaluaciones ({DAYS_BACK} días)")
    print(f"{'='*70}")
    print(f"Total evaluaciones: {total_evals}")
    print(f"Aprobadas: {passed_evals} ({pass_rate:.1f}%)")
    print(f"Suspendidas: {total_evals - passed_evals} ({100-pass_rate:.1f}%)")
    print(f"Estudiantes únicos: {len(student_stats)}")
    print(f"Media evaluaciones/estudiante: {total_evals/len(student_stats):.1f}")
    print(f"Mejor rendimiento individual: {max(student_rates):.1f}%")
    print(f"Peor rendimiento individual: {min(student_rates):.1f}%")

⚠️ No hay evaluaciones en el período seleccionado


In [15]:
# Tiempos de Respuesta y Eficiencia

# Obtener interacciones con tiempos
all_interactions = list(interactions.find({
    "timestamp": {"$gte": start_date, "$lte": end_date},
    "response_time_seconds": {"$exists": True}
}))

if not all_interactions:
    print("⚠️ No hay interacciones con tiempos registrados")
else:
    # Separar por tipo
    chat_times = [i['response_time_seconds'] for i in all_interactions if i.get('type') == 'chat_interaction']
    eval_times = [i['response_time_seconds'] for i in all_interactions if i.get('type') == 'evaluation']
    
    # Correlación tiempo-aprobación
    eval_pass_times = [i['response_time_seconds'] for i in all_interactions 
                      if i.get('type') == 'evaluation' and i.get('evaluation_result', {}).get('pass')]
    eval_fail_times = [i['response_time_seconds'] for i in all_interactions 
                      if i.get('type') == 'evaluation' and not i.get('evaluation_result', {}).get('pass')]
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Tiempos de Respuesta', fontsize=16, fontweight='bold')
    
    # 1. Comparación por tipo
    ax1 = axes[0, 0]
    if chat_times and eval_times:
        data_to_plot = [chat_times, eval_times]
        bp = ax1.boxplot(data_to_plot, labels=['Chat', 'Evaluaciones'], patch_artist=True)
        for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
            patch.set_facecolor(color)
        ax1.set_ylabel('Tiempo de Respuesta (segundos)')
        ax1.set_title('Comparación de Tiempos por Tipo')
        ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Distribución de tiempos en evaluaciones
    ax2 = axes[0, 1]
    if eval_times:
        ax2.hist(eval_times, bins=20, color='#9b59b6', edgecolor='black', alpha=0.7)
        ax2.axvline(np.median(eval_times), color='red', linestyle='--', 
                    label=f'Mediana: {np.median(eval_times):.2f}s')
        ax2.set_xlabel('Tiempo de Respuesta (segundos)')
        ax2.set_ylabel('Frecuencia')
        ax2.set_title('Distribución de Tiempos en Evaluaciones')
        ax2.legend()
    
    # 3. Tiempo vs Resultado
    ax3 = axes[1, 0]
    if eval_pass_times and eval_fail_times:
        bp = ax3.boxplot([eval_pass_times, eval_fail_times], 
                   labels=['Aprobados', 'Suspendidos'],
                   patch_artist=True)
        for patch, color in zip(bp['boxes'], ['#2ecc71', '#e74c3c']):
            patch.set_facecolor(color)
        ax3.set_ylabel('Tiempo de Respuesta (segundos)')
        ax3.set_title('Tiempo de Respuesta vs Resultado')
        ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Evolución temporal
    ax4 = axes[1, 1]
    daily_times = defaultdict(list)
    for interaction in all_interactions:
        if interaction.get('type') == 'evaluation':
            date = interaction['timestamp'].date()
            daily_times[date].append(interaction['response_time_seconds'])
    
    if daily_times:
        sorted_dates = sorted(daily_times.keys())
        avg_times = [np.mean(daily_times[d]) for d in sorted_dates]
        
        ax4.plot(sorted_dates, avg_times, marker='o', linewidth=2, color='#1abc9c')
        ax4.set_xlabel('Fecha')
        ax4.set_ylabel('Tiempo Medio (segundos)')
        ax4.set_title('Evolución del Tiempo Medio de Evaluación')
        ax4.grid(True, alpha=0.3)
        plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Tiempos de Respuesta")
    print(f"{'='*70}")
    if chat_times:
        print(f"Chat:")
        print(f"  Media: {np.mean(chat_times):.2f}s")
        print(f"  Mediana: {np.median(chat_times):.2f}s")
        print(f"  Min: {min(chat_times):.2f}s | Max: {max(chat_times):.2f}s")
    
    if eval_times:
        print(f"\nEvaluaciones:")
        print(f"  Media: {np.mean(eval_times):.2f}s")
        print(f"  Mediana: {np.median(eval_times):.2f}s")
        print(f"  Min: {min(eval_times):.2f}s | Max: {max(eval_times):.2f}s")
    
    if eval_pass_times and eval_fail_times:
        print(f"\nComparación Aprobados vs Suspendidos:")
        print(f"  Aprobados - Media: {np.mean(eval_pass_times):.2f}s")
        print(f"  Suspendidos - Media: {np.mean(eval_fail_times):.2f}s")
        diff = np.mean(eval_fail_times) - np.mean(eval_pass_times)
        print(f"  Diferencia: {diff:+.2f}s")

⚠️ No hay interacciones con tiempos registrados


In [16]:
# Patrones de Uso y Engagement

# Obtener todas las interacciones
all_interactions = list(interactions.find({
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not all_interactions:
    print("⚠️ No hay interacciones en el período")
else:
    # Análisis por hora
    hour_counts = Counter(i['timestamp'].hour for i in all_interactions)
    
    # Análisis por día de semana
    weekday_counts = Counter(i['timestamp'].weekday() for i in all_interactions)
    
    # Sesiones por usuario
    user_sessions = defaultdict(set)
    for i in all_interactions:
        user_sessions[i.get('user_id')].add(i.get('session_id'))
    
    # Interacciones por usuario
    user_interactions = Counter(i.get('user_id') for i in all_interactions)
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Patrones de Uso', fontsize=16, fontweight='bold')
    
    # 1. Actividad por hora
    ax1 = axes[0, 0]
    hours = sorted(hour_counts.keys())
    counts = [hour_counts[h] for h in hours]
    ax1.bar(hours, counts, color='#3498db', edgecolor='black')
    ax1.set_xlabel('Hora del Día')
    ax1.set_ylabel('Número de Interacciones')
    ax1.set_title('Distribución de Actividad por Hora')
    ax1.set_xticks(range(24))
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Actividad por día de semana
    ax2 = axes[0, 1]
    weekdays = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
    weekday_data = [weekday_counts[i] for i in range(7)]
    colors = ['#e74c3c' if i >= 5 else '#2ecc71' for i in range(7)]
    ax2.bar(weekdays, weekday_data, color=colors, edgecolor='black')
    ax2.set_xlabel('Día de la Semana')
    ax2.set_ylabel('Número de Interacciones')
    ax2.set_title('Distribución por Día (Verde=Lab, Rojo=Fin Semana)')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Sesiones por estudiante
    ax3 = axes[1, 0]
    sessions_per_user = [len(sessions) for sessions in user_sessions.values()]
    ax3.hist(sessions_per_user, bins=15, color='#9b59b6', edgecolor='black', alpha=0.7)
    ax3.axvline(np.mean(sessions_per_user), color='red', linestyle='--',
               label=f'Media: {np.mean(sessions_per_user):.1f}')
    ax3.set_xlabel('Número de Sesiones')
    ax3.set_ylabel('Número de Estudiantes')
    ax3.set_title('Distribución de Sesiones por Estudiante')
    ax3.legend()
    
    # 4. Top usuarios
    ax4 = axes[1, 1]
    top_users = user_interactions.most_common(10)
    user_labels = [f"Est. {i+1}" for i in range(len(top_users))]
    user_counts = [count for _, count in top_users]
    
    ax4.barh(user_labels, user_counts, color='#e67e22')
    ax4.set_xlabel('Número de Interacciones')
    ax4.set_title('Top 10 Usuarios Más Activos')
    ax4.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Patrones de Uso")
    print(f"{'='*70}")
    print(f"Total interacciones: {len(all_interactions)}")
    print(f"Usuarios únicos: {len(user_sessions)}")
    print(f"Sesiones únicas: {sum(len(s) for s in user_sessions.values())}")
    print(f"Media interacciones/usuario: {len(all_interactions)/len(user_sessions):.1f}")
    print(f"Hora pico: {max(hour_counts, key=hour_counts.get)}:00 ({hour_counts[max(hour_counts, key=hour_counts.get)]} interacciones)")
    peak_day = max(weekday_counts, key=weekday_counts.get)
    print(f"Día más activo: {weekdays[peak_day]} ({weekday_counts[peak_day]} interacciones)")

⚠️ No hay interacciones en el período


In [17]:
# Contenido y Engagement

# Obtener interacciones
all_interactions = list(interactions.find({
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not all_interactions:
    print("⚠️ No hay interacciones")
else:
    # Vídeos más utilizados
    video_counts = Counter(i.get('video_url') for i in all_interactions if i.get('video_url'))
    
    # Tasa de éxito por vídeo
    video_success = defaultdict(lambda: {'passed': 0, 'total': 0})
    for i in all_interactions:
        if i.get('type') == 'evaluation' and i.get('video_url'):
            video_url = i['video_url']
            video_success[video_url]['total'] += 1
            if i.get('evaluation_result', {}).get('pass'):
                video_success[video_url]['passed'] += 1
    
    # Idiomas y modelos
    language_counts = Counter(i.get('language') for i in all_interactions if i.get('language'))
    model_counts = Counter(i.get('model_chat') for i in all_interactions if i.get('model_chat'))
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Contenido y Engagement', fontsize=16, fontweight='bold')
    
    # 1. Top vídeos
    ax1 = axes[0, 0]
    if video_counts:
        top_videos = video_counts.most_common(10)
        video_labels = [f"Video {i+1}" for i in range(len(top_videos))]
        video_views = [count for _, count in top_videos]
        
        ax1.barh(video_labels, video_views, color='#3498db')
        ax1.set_xlabel('Número de Interacciones')
        ax1.set_title('Top 10 Vídeos Más Utilizados')
        ax1.invert_yaxis()
    
    # 2. Tasa de éxito por vídeo (min 3 evaluaciones)
    ax2 = axes[0, 1]
    if video_success:
        video_success_rates = [(v, (stats['passed'] / stats['total']) * 100) 
                              for v, stats in video_success.items() if stats['total'] >= 3]
        video_success_rates.sort(key=lambda x: x[1], reverse=True)
        
        if video_success_rates:
            top_success_videos = video_success_rates[:10]
            labels = [f"Video {i+1}" for i in range(len(top_success_videos))]
            rates = [rate for _, rate in top_success_videos]
            
            ax2.barh(labels, rates, color='#2ecc71')
            ax2.set_xlabel('Tasa de Aprobación (%)')
            ax2.set_title('Top 10 Vídeos por Tasa de Éxito (min. 3 eval.)')
            ax2.invert_yaxis()
    
    # 3. Distribución de idiomas
    ax3 = axes[1, 0]
    if language_counts:
        langs = list(language_counts.keys())
        counts = list(language_counts.values())
        colors_lang = plt.cm.Set3(range(len(langs)))
        
        ax3.pie(counts, labels=langs, autopct='%1.1f%%', colors=colors_lang, startangle=90)
        ax3.set_title('Distribución de Idiomas')
    
    # 4. Modelos utilizados
    ax4 = axes[1, 1]
    if model_counts:
        models = list(model_counts.keys())
        counts = list(model_counts.values())
        model_labels = [m.split('-')[0] if m else 'N/A' for m in models]
        
        ax4.bar(range(len(models)), counts, color='#e67e22', edgecolor='black')
        ax4.set_xticks(range(len(models)))
        ax4.set_xticklabels(model_labels, rotation=45, ha='right')
        ax4.set_ylabel('Número de Interacciones')
        ax4.set_title('Modelos Más Utilizados')
        ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Contenido y Engagement")
    print(f"{'='*70}")
    print(f"Vídeos únicos utilizados: {len(video_counts)}")
    print(f"Total interacciones con vídeo: {sum(video_counts.values())}")
    if language_counts:
        top_lang = language_counts.most_common(1)[0]
        print(f"Idioma principal: {top_lang[0]} ({top_lang[1]} interacciones)")
    if model_counts:
        top_model = model_counts.most_common(1)[0]
        print(f"Modelo más usado: {top_model[0]} ({top_model[1]} interacciones)")

⚠️ No hay interacciones


In [18]:
# Patrones de Acceso y Seguridad

# Obtener hashes
hashes = list(access_hashes.find({
    "created_at": {"$gte": start_date, "$lte": end_date}
}))

if not hashes:
    print("⚠️ No hay hashes en el período")
else:
    # Métricas
    used_hashes = sum(1 for h in hashes if h.get('used'))
    unused_hashes = len(hashes) - used_hashes
    
    # Distribución temporal
    daily_hashes = defaultdict(lambda: {'created': 0, 'used': 0})
    for h in hashes:
        date = h['created_at'].date()
        daily_hashes[date]['created'] += 1
        if h.get('used'):
            daily_hashes[date]['used'] += 1
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Accesos y Seguridad', fontsize=16, fontweight='bold')
    
    # 1. Estado de hashes
    ax1 = axes[0, 0]
    sizes = [used_hashes, unused_hashes]
    colors = ['#2ecc71', '#e74c3c']
    labels = [f'Utilizados\n({used_hashes})', f'No utilizados\n({unused_hashes})']
    ax1.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    ax1.set_title('Estado de Hashes de Acceso')
    
    # 2. Evolución de creación
    ax2 = axes[0, 1]
    sorted_dates = sorted(daily_hashes.keys())
    created_counts = [daily_hashes[d]['created'] for d in sorted_dates]
    
    ax2.plot(sorted_dates, created_counts, marker='o', linewidth=2, color='#3498db')
    ax2.fill_between(sorted_dates, created_counts, alpha=0.3, color='#3498db')
    ax2.set_xlabel('Fecha')
    ax2.set_ylabel('Hashes Creados')
    ax2.set_title('Evolución de Creación de Hashes')
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    
    # 3. Tasa de uso diaria
    ax3 = axes[1, 0]
    usage_rates = [(daily_hashes[d]['used'] / daily_hashes[d]['created']) * 100 
                   for d in sorted_dates if daily_hashes[d]['created'] > 0]
    
    if usage_rates:
        ax3.bar(range(len(usage_rates)), usage_rates, color='#9b59b6', edgecolor='black')
        ax3.axhline(np.mean(usage_rates), color='red', linestyle='--',
                   label=f'Media: {np.mean(usage_rates):.1f}%')
        ax3.set_xlabel('Días')
        ax3.set_ylabel('Tasa de Uso (%)')
        ax3.set_title('Tasa de Uso Diaria de Hashes')
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Resumen textual
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    stats_text = f"""
    ESTADÍSTICAS DE ACCESO
    {'='*40}
    
    Total hashes creados: {len(hashes)}
    Hashes utilizados: {used_hashes} ({(used_hashes/len(hashes)*100):.1f}%)
    Hashes sin usar: {unused_hashes} ({(unused_hashes/len(hashes)*100):.1f}%)
    
    Período: {DAYS_BACK} días
    Inicio: {start_date.strftime('%Y-%m-%d')}
    Fin: {end_date.strftime('%Y-%m-%d')}
    
    Media hashes/día: {len(hashes)/DAYS_BACK:.1f}
    """
    
    ax4.text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
            verticalalignment='center')
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Accesos y Seguridad")
    print(f"{'='*70}")
    print(f"Total hashes: {len(hashes)}")
    print(f"Utilizados: {used_hashes} ({(used_hashes/len(hashes)*100):.1f}%)")
    print(f"Sin usar: {unused_hashes} ({(unused_hashes/len(hashes)*100):.1f}%)")
    print(f"Media creación/día: {len(hashes)/DAYS_BACK:.1f}")

⚠️ No hay hashes en el período


In [19]:
# Vista General Completa

# Recopilar todos los datos
all_interactions = list(interactions.find({
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not all_interactions:
    print("⚠️ No hay datos para el dashboard")
else:
    evaluations = [i for i in all_interactions if i.get('type') == 'evaluation']
    chats = [i for i in all_interactions if i.get('type') == 'chat_interaction']
    
    # KPIs principales
    total_students = len(set(i.get('user_id') for i in all_interactions))
    total_sessions = len(set(i.get('session_id') for i in all_interactions))
    pass_rate = (sum(1 for e in evaluations if e.get('evaluation_result', {}).get('pass', False)) 
                / len(evaluations) * 100) if evaluations else 0
    avg_response_time = np.mean([i['response_time_seconds'] for i in all_interactions 
                                if 'response_time_seconds' in i]) if all_interactions else 0
    
    # VISUALIZACIÓN
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
    
    fig.suptitle(f'Dashboard de Rendimiento Estudiantil - Últimos {DAYS_BACK} días', 
                fontsize=18, fontweight='bold')
    
    # KPIs (primera fila)
    kpi_data = [
        ('Estudiantes\nActivos', total_students, '#3498db'),
        ('Sesiones\nTotales', total_sessions, '#2ecc71'),
        ('Tasa de\nAprobación', f"{pass_rate:.1f}%", '#e74c3c'),
        ('Tiempo Medio\nRespuesta', f"{avg_response_time:.1f}s", '#9b59b6')
    ]
    
    for i, (title, value, color) in enumerate(kpi_data):
        ax = fig.add_subplot(gs[0, i])
        ax.text(0.5, 0.5, str(value), ha='center', va='center', 
               fontsize=36, fontweight='bold', color=color)
        ax.text(0.5, 0.2, title, ha='center', va='center', 
               fontsize=12, color='gray')
        ax.axis('off')
    
    # Actividad diaria
    ax1 = fig.add_subplot(gs[1, :2])
    daily_activity = defaultdict(int)
    for i in all_interactions:
        daily_activity[i['timestamp'].date()] += 1
    dates = sorted(daily_activity.keys())
    counts = [daily_activity[d] for d in dates]
    ax1.plot(dates, counts, marker='o', linewidth=2, color='#1abc9c')
    ax1.fill_between(dates, counts, alpha=0.3, color='#1abc9c')
    ax1.set_title('Actividad Diaria', fontweight='bold')
    ax1.set_xlabel('Fecha')
    ax1.set_ylabel('Interacciones')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    
    # Tipo de interacciones
    ax2 = fig.add_subplot(gs[1, 2:])
    type_counts = Counter(i.get('type') for i in all_interactions)
    ax2.bar(type_counts.keys(), type_counts.values(), 
           color=['#3498db', '#e74c3c'], edgecolor='black')
    ax2.set_title('Distribución de Tipos', fontweight='bold')
    ax2.set_ylabel('Cantidad')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Evolución tasa de aprobación
    ax3 = fig.add_subplot(gs[2, :2])
    daily_evals = defaultdict(lambda: {'passed': 0, 'total': 0})
    for e in evaluations:
        date = e['timestamp'].date()
        daily_evals[date]['total'] += 1
        if e.get('evaluation_result', {}).get('pass'):
            daily_evals[date]['passed'] += 1
    
    eval_dates = sorted(daily_evals.keys())
    pass_rates = [(daily_evals[d]['passed'] / daily_evals[d]['total']) * 100 
                 for d in eval_dates if daily_evals[d]['total'] > 0]
    
    if pass_rates:
        ax3.plot(eval_dates[:len(pass_rates)], pass_rates, marker='o', 
                linewidth=2, color='#e67e22')
        ax3.axhline(pass_rate, color='gray', linestyle='--', alpha=0.5, label='Media')
        ax3.fill_between(eval_dates[:len(pass_rates)], pass_rates, alpha=0.3, color='#e67e22')
        ax3.set_title('Evolución Tasa de Aprobación', fontweight='bold')
        ax3.set_xlabel('Fecha')
        ax3.set_ylabel('Tasa (%)')
        ax3.grid(True, alpha=0.3)
        ax3.legend()
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
    
    # Top estudiantes
    ax4 = fig.add_subplot(gs[2, 2:])
    user_activity = Counter(i.get('user_id') for i in all_interactions)
    top_users = user_activity.most_common(8)
    users = [f"Est. {i+1}" for i in range(len(top_users))]
    activities = [count for _, count in top_users]
    
    ax4.barh(users, activities, color='#9b59b6')
    ax4.set_title('Top Usuarios Activos', fontweight='bold')
    ax4.set_xlabel('Interacciones')
    ax4.invert_yaxis()
    
    plt.show()
    
    # RESUMEN GLOBAL
    print(f"\n{'='*70}")
    print(f"RESUMEN GLOBAL - Dashboard ({DAYS_BACK} días)")
    print(f"{'='*70}")
    print(f"Período: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")
    print(f"\nActividad:")
    print(f"  Total interacciones: {len(all_interactions)}")
    print(f"  Evaluaciones: {len(evaluations)}")
    print(f"  Chats: {len(chats)}")
    print(f"\nUsuarios:")
    print(f"  Estudiantes activos: {total_students}")
    print(f"  Sesiones totales: {total_sessions}")
    print(f"\nRendimiento:")
    print(f"  Tasa aprobación: {pass_rate:.1f}%")
    print(f"  Tiempo medio respuesta: {avg_response_time:.1f}s")

⚠️ No hay datos para el dashboard


In [20]:
# Progresión de Aprendizaje Individual

# Obtener evaluaciones ordenadas por estudiante y tiempo
evaluations = list(interactions.find({
    "type": "evaluation",
    "timestamp": {"$gte": start_date, "$lte": end_date}
}).sort([("user_id", 1), ("timestamp", 1)]))

if not evaluations:
    print("⚠️ No hay evaluaciones")
else:
    # Agrupar por estudiante
    student_progress = defaultdict(list)
    for eval in evaluations:
        user_id = eval.get('user_id')
        student_progress[user_id].append({
            'timestamp': eval['timestamp'],
            'passed': eval.get('evaluation_result', {}).get('pass', False),
            'time': eval.get('response_time_seconds', 0)
        })
    
    # Filtrar estudiantes con al menos 5 evaluaciones
    active_students = {uid: evals for uid, evals in student_progress.items() if len(evals) >= 5}
    
    if not active_students:
        print("⚠️ No hay estudiantes con suficientes evaluaciones (min. 5)")
    else:
        # VISUALIZACIÓN
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Análisis de Progresión de Aprendizaje', fontsize=16, fontweight='bold')
        
        # 1. Curvas de aprendizaje (primeros 5 estudiantes)
        ax1 = axes[0, 0]
        colors = plt.cm.tab10(range(5))
        for i, (uid, evals) in enumerate(list(active_students.items())[:5]):
            # Media móvil de éxito
            success = [1 if e['passed'] else 0 for e in evals]
            window = min(3, len(success))
            moving_avg = pd.Series(success).rolling(window=window).mean()
            ax1.plot(range(len(moving_avg)), moving_avg * 100, 
                    marker='o', label=f'Est. {i+1}', color=colors[i])
        
        ax1.set_xlabel('Número de Evaluación')
        ax1.set_ylabel('Tasa de Éxito (%) - Media Móvil')
        ax1.set_title('Curvas de Aprendizaje (Top 5 estudiantes activos)')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Mejora entre primera y última mitad
        ax2 = axes[0, 1]
        improvements = []
        labels = []
        for i, (uid, evals) in enumerate(list(active_students.items())[:10]):
            mid = len(evals) // 2
            first_half = sum(1 for e in evals[:mid] if e['passed']) / mid * 100
            second_half = sum(1 for e in evals[mid:] if e['passed']) / (len(evals) - mid) * 100
            improvement = second_half - first_half
            improvements.append(improvement)
            labels.append(f'Est. {i+1}')
        
        colors_imp = ['#2ecc71' if x >= 0 else '#e74c3c' for x in improvements]
        ax2.barh(labels, improvements, color=colors_imp)
        ax2.axvline(0, color='black', linestyle='-', linewidth=0.5)
        ax2.set_xlabel('Mejora (%)')
        ax2.set_title('Mejora: 2ª Mitad vs 1ª Mitad')
        ax2.invert_yaxis()
        
        # 3. Reducción de tiempo con experiencia
        ax3 = axes[1, 0]
        for i, (uid, evals) in enumerate(list(active_students.items())[:5]):
            times = [e['time'] for e in evals if e['time'] > 0]
            if len(times) >= 3:
                ax3.plot(range(len(times)), times, marker='o', 
                        label=f'Est. {i+1}', color=colors[i], alpha=0.7)
        
        ax3.set_xlabel('Número de Evaluación')
        ax3.set_ylabel('Tiempo de Respuesta (s)')
        ax3.set_title('Evolución del Tiempo de Respuesta')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Distribución de rachas
        ax4 = axes[1, 1]
        max_streaks = []
        for uid, evals in active_students.items():
            current_streak = 0
            max_streak = 0
            for e in evals:
                if e['passed']:
                    current_streak += 1
                    max_streak = max(max_streak, current_streak)
                else:
                    current_streak = 0
            max_streaks.append(max_streak)
        
        ax4.hist(max_streaks, bins=range(max(max_streaks)+2), 
                color='#9b59b6', edgecolor='black', alpha=0.7)
        ax4.set_xlabel('Racha Máxima de Aprobados')
        ax4.set_ylabel('Número de Estudiantes')
        ax4.set_title('Distribución de Rachas Positivas')
        
        plt.tight_layout()
        plt.show()
        
        # RESUMEN
        print(f"\n{'='*70}")
        print(f"RESUMEN - Progresión de Aprendizaje")
        print(f"{'='*70}")
        print(f"Estudiantes con 5+ evaluaciones: {len(active_students)}")
        print(f"Mejora promedio (2ª vs 1ª mitad): {np.mean(improvements):.1f}%")
        print(f"Estudiantes con mejora: {sum(1 for x in improvements if x > 0)}/{len(improvements)}")
        print(f"Racha máxima promedio: {np.mean(max_streaks):.1f}")

⚠️ No hay evaluaciones


In [21]:
# Interacciones de Chat y Preguntas

# Obtener chats
chats = list(interactions.find({
    "type": "chat_interaction",
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not chats:
    print("⚠️ No hay interacciones de chat")
else:
    # Análisis de longitud de preguntas
    question_lengths = [len(c.get('user_input', '')) for c in chats if c.get('user_input')]
    response_lengths = [len(c.get('model_response', '')) for c in chats if c.get('model_response')]
    
    # Palabras clave más comunes (simple análisis)
    all_words = []
    for chat in chats:
        user_input = chat.get('user_input', '').lower()
        # Palabras simples > 4 caracteres
        words = [w for w in user_input.split() if len(w) > 4]
        all_words.extend(words)
    
    word_freq = Counter(all_words)
    
    # Chats por estudiante
    chats_per_user = Counter(c.get('user_id') for c in chats)
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Análisis de Interacciones de Chat', fontsize=16, fontweight='bold')
    
    # 1. Distribución de longitud de preguntas
    ax1 = axes[0, 0]
    ax1.hist(question_lengths, bins=30, color='#3498db', edgecolor='black', alpha=0.7)
    ax1.axvline(np.median(question_lengths), color='red', linestyle='--',
               label=f'Mediana: {np.median(question_lengths):.0f} chars')
    ax1.set_xlabel('Longitud de Pregunta (caracteres)')
    ax1.set_ylabel('Frecuencia')
    ax1.set_title('Distribución de Longitud de Preguntas')
    ax1.legend()
    
    # 2. Palabras clave más comunes
    ax2 = axes[0, 1]
    top_words = word_freq.most_common(15)
    if top_words:
        words = [w[0] for w in top_words]
        counts = [w[1] for w in top_words]
        ax2.barh(words, counts, color='#2ecc71')
        ax2.set_xlabel('Frecuencia')
        ax2.set_title('Top 15 Palabras en Preguntas (>4 chars)')
        ax2.invert_yaxis()
    
    # 3. Ratio pregunta/respuesta
    ax3 = axes[1, 0]
    ratios = [r/q if q > 0 else 0 for q, r in zip(question_lengths, response_lengths)]
    ax3.scatter(question_lengths, response_lengths, alpha=0.5, color='#9b59b6')
    ax3.set_xlabel('Longitud Pregunta (chars)')
    ax3.set_ylabel('Longitud Respuesta (chars)')
    ax3.set_title('Relación Pregunta-Respuesta')
    ax3.grid(True, alpha=0.3)
    
    # 4. Top usuarios de chat
    ax4 = axes[1, 1]
    top_chat_users = chats_per_user.most_common(10)
    users = [f"Est. {i+1}" for i in range(len(top_chat_users))]
    counts = [c for _, c in top_chat_users]
    
    ax4.barh(users, counts, color='#e67e22')
    ax4.set_xlabel('Número de Preguntas')
    ax4.set_title('Top 10 Usuarios por Preguntas en Chat')
    ax4.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN
    print(f"\n{'='*70}")
    print(f"RESUMEN - Interacciones de Chat")
    print(f"{'='*70}")
    print(f"Total preguntas: {len(chats)}")
    print(f"Usuarios únicos: {len(chats_per_user)}")
    print(f"Media preguntas/usuario: {len(chats)/len(chats_per_user):.1f}")
    print(f"Longitud media pregunta: {np.mean(question_lengths):.0f} chars")
    print(f"Longitud media respuesta: {np.mean(response_lengths):.0f} chars")
    if top_words:
        print(f"Palabra más frecuente: '{top_words[0][0]}' ({top_words[0][1]} veces)")

⚠️ No hay interacciones de chat


In [22]:
# Detección de Riesgo y Estudiantes que Necesitan Ayuda

# Obtener datos por estudiante
all_interactions = list(interactions.find({
    "timestamp": {"$gte": start_date, "$lte": end_date}
}))

if not all_interactions:
    print("⚠️ No hay datos")
else:
    # Analizar por estudiante
    student_metrics = defaultdict(lambda: {
        'total_evals': 0,
        'passed_evals': 0,
        'failed_evals': 0,
        'consecutive_fails': 0,
        'max_consecutive_fails': 0,
        'total_chats': 0,
        'avg_time': [],
        'last_activity': None,
        'total_sessions': set()
    })
    
    # Recopilar métricas
    for i in all_interactions:
        uid = i.get('user_id')
        student_metrics[uid]['total_sessions'].add(i.get('session_id'))
        student_metrics[uid]['last_activity'] = max(
            student_metrics[uid]['last_activity'] or i['timestamp'], 
            i['timestamp']
        )
        
        if i.get('type') == 'evaluation':
            student_metrics[uid]['total_evals'] += 1
            if 'response_time_seconds' in i:
                student_metrics[uid]['avg_time'].append(i['response_time_seconds'])
            
            if i.get('evaluation_result', {}).get('pass'):
                student_metrics[uid]['passed_evals'] += 1
                student_metrics[uid]['consecutive_fails'] = 0
            else:
                student_metrics[uid]['failed_evals'] += 1
                student_metrics[uid]['consecutive_fails'] += 1
                student_metrics[uid]['max_consecutive_fails'] = max(
                    student_metrics[uid]['max_consecutive_fails'],
                    student_metrics[uid]['consecutive_fails']
                )
        elif i.get('type') == 'chat_interaction':
            student_metrics[uid]['total_chats'] += 1
    
    # Calcular score de riesgo
    risk_students = []
    for uid, metrics in student_metrics.items():
        if metrics['total_evals'] < 3:
            continue
            
        risk_score = 0
        risk_factors = []
        
        # Factor 1: Tasa de aprobación baja
        pass_rate = metrics['passed_evals'] / metrics['total_evals'] * 100
        if pass_rate < 40:
            risk_score += 3
            risk_factors.append(f"Tasa aprobación: {pass_rate:.0f}%")
        elif pass_rate < 60:
            risk_score += 2
            risk_factors.append(f"Tasa aprobación: {pass_rate:.0f}%")
        
        # Factor 2: Rachas de fallos
        if metrics['max_consecutive_fails'] >= 3:
            risk_score += 3
            risk_factors.append(f"Racha de {metrics['max_consecutive_fails']} fallos")
        elif metrics['max_consecutive_fails'] >= 2:
            risk_score += 1
        
        # Factor 3: Inactividad reciente
        days_since_activity = (datetime.now() - metrics['last_activity']).days
        if days_since_activity > 7:
            risk_score += 2
            risk_factors.append(f"Inactivo {days_since_activity} días")
        elif days_since_activity > 3:
            risk_score += 1
        
        # Factor 4: Pocas preguntas (baja participación)
        if metrics['total_evals'] > 0:
            chat_eval_ratio = metrics['total_chats'] / metrics['total_evals']
            if chat_eval_ratio < 0.2:
                risk_score += 1
                risk_factors.append(f"Baja participación chat")
        
        if risk_score > 0:
            risk_students.append({
                'user_id': uid,
                'risk_score': risk_score,
                'pass_rate': pass_rate,
                'total_evals': metrics['total_evals'],
                'consecutive_fails': metrics['max_consecutive_fails'],
                'days_inactive': days_since_activity,
                'risk_factors': risk_factors
            })
    
    # Ordenar por riesgo
    risk_students.sort(key=lambda x: x['risk_score'], reverse=True)
    
    # VISUALIZACIÓN
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Detección de Estudiantes en Riesgo', fontsize=16, fontweight='bold')
    
    # 1. Distribución de scores de riesgo
    ax1 = axes[0, 0]
    risk_scores = [s['risk_score'] for s in risk_students]
    if risk_scores:
        ax1.hist(risk_scores, bins=range(max(risk_scores)+2), 
                color='#e74c3c', edgecolor='black', alpha=0.7)
        ax1.set_xlabel('Score de Riesgo')
        ax1.set_ylabel('Número de Estudiantes')
        ax1.set_title('Distribución de Riesgo')
    
    # 2. Top estudiantes en riesgo
    ax2 = axes[0, 1]
    top_risk = risk_students[:10]
    if top_risk:
        labels = [f"Est. {i+1}\n({s['risk_score']})" for i, s in enumerate(top_risk)]
        scores = [s['risk_score'] for s in top_risk]
        colors_risk = ['#e74c3c' if s >= 5 else '#e67e22' if s >= 3 else '#f39c12' 
                      for s in scores]
        
        ax2.barh(labels, scores, color=colors_risk)
        ax2.set_xlabel('Score de Riesgo')
        ax2.set_title('Top 10 Estudiantes en Riesgo')
        ax2.invert_yaxis()
    
    # 3. Correlación tasa aprobación vs inactividad
    ax3 = axes[1, 0]
    if risk_students:
        pass_rates = [s['pass_rate'] for s in risk_students]
        inactivity = [s['days_inactive'] for s in risk_students]
        colors_scatter = [s['risk_score'] for s in risk_students]
        
        scatter = ax3.scatter(inactivity, pass_rates, c=colors_scatter, 
                            cmap='RdYlGn_r', alpha=0.6, s=100)
        ax3.set_xlabel('Días de Inactividad')
        ax3.set_ylabel('Tasa de Aprobación (%)')
        ax3.set_title('Relación Inactividad vs Rendimiento')
        ax3.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax3, label='Risk Score')
    
    # 4. Categorización de riesgo
    ax4 = axes[1, 1]
    high_risk = sum(1 for s in risk_students if s['risk_score'] >= 5)
    medium_risk = sum(1 for s in risk_students if 3 <= s['risk_score'] < 5)
    low_risk = sum(1 for s in risk_students if s['risk_score'] < 3)
    
    categories = ['Alto\nRiesgo', 'Medio\nRiesgo', 'Bajo\nRiesgo']
    counts = [high_risk, medium_risk, low_risk]
    colors_cat = ['#e74c3c', '#e67e22', '#f39c12']
    
    ax4.bar(categories, counts, color=colors_cat, edgecolor='black')
    ax4.set_ylabel('Número de Estudiantes')
    ax4.set_title('Categorización de Riesgo')
    for i, v in enumerate(counts):
        ax4.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # RESUMEN Y ALERTAS
    print(f"\n{'='*70}")
    print(f"DETECCIÓN DE RIESGO - ALERTAS")
    print(f"{'='*70}")
    print(f"Total estudiantes analizados: {len(student_metrics)}")
    print(f"Estudiantes con riesgo detectado: {len(risk_students)}")
    print(f"  - Alto riesgo (≥5): {high_risk}")
    print(f"  - Medio riesgo (3-4): {medium_risk}")
    print(f"  - Bajo riesgo (<3): {low_risk}")
    
    if top_risk:
        print(f"\n🚨 TOP 5 ESTUDIANTES QUE NECESITAN INTERVENCIÓN:")
        print(f"{'-'*70}")
        for i, student in enumerate(top_risk[:5], 1):
            print(f"\n{i}. Estudiante (ID: ...{student['user_id'][-8:]})")
            print(f"   Score de Riesgo: {student['risk_score']}/10")
            print(f"   Factores:")
            for factor in student['risk_factors']:
                print(f"     • {factor}")

⚠️ No hay datos
